In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [0]:
data = [
    (101, "Electronics", "1200", "Phone", 1000, "North", "High"),
    (102, "Furniture", "2500", "Table", 2000, "South", "Low"),
    (103, "Electronics", "800", "Laptop", 700, "West", "Medium"),
    (104, "Clothing", "500", "Shirt", 400, "North", "High"),
    (105, "Electronics", None, "Speaker", 1500, "East", "High")
]

columns = [
    "product_id",
    "category",
    "price",
    "product_name",
    "base_price",
    "region",
    "priority"
]

df = spark.createDataFrame(data, columns)

df.show()

+----------+-----------+-----+------------+----------+------+--------+
|product_id|   category|price|product_name|base_price|region|priority|
+----------+-----------+-----+------------+----------+------+--------+
|       101|Electronics| 1200|       Phone|      1000| North|    High|
|       102|  Furniture| 2500|       Table|      2000| South|     Low|
|       103|Electronics|  800|      Laptop|       700|  West|  Medium|
|       104|   Clothing|  500|       Shirt|       400| North|    High|
|       105|Electronics| NULL|     Speaker|      1500|  East|    High|
+----------+-----------+-----+------------+----------+------+--------+



In [0]:
orders = [
    (1, "Completed", 1500),
    (2, "Pending", 800),
    (3, "Completed", 500),
    (4, "Completed", 2200),
    (5, "Cancelled", 1200)
]

df_orders = spark.createDataFrame(
    orders,
    ["order_id", "status", "amount"]
)

df_orders.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1500|
|       2|  Pending|   800|
|       3|Completed|   500|
|       4|Completed|  2200|
|       5|Cancelled|  1200|
+--------+---------+------+



** 
## Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application? **

ANS: The roles of the Driver, Cluster Manager, and Executor in a Spark application are as follows :

Driver

- Controls the Spark application.
- Creates execution plan (DAG).
- Sends tasks to executors.
- Collects results.

Cluster Manager

- Allocates CPU and memory.
- Launches executors.
- Examples: Standalone, YARN, Kubernetes.

Executor

- Executes tasks.
- Stores intermediate data.
- Returns results to Driver.

 **
## Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets? **


ANS: Spark records all transformations without executing them immediately.

Execution starts only when an Action like show(), count(), or write() is called.

Advantages:

Optimized execution
Reduced disk I/O
Faster processing
Better resource utilization**

**
## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.
**

df_csv = spark.read.option("header", True).option("inferSchema",  True).csv("data/source.csv")

df_csv.show()

**
## Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?
**


| CSV            | Parquet       |
| -------------- | ------------- |
| Row-based      | Column-based  |
| Text file      | Binary file   |
| Larger storage | Compressed    |
| Slower         | Faster        |
| No schema      | Schema stored |

Insight: Parquet is faster because it reads only required columns and supports compression and Predicate Pushdown.

In [0]:
# Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

result = df.filter(col("category") == "Electronics") \
           .select("product_id", "price")

result.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101| 1200|
|       103|  800|
|       105| NULL|
+----------+-----+



In [0]:
# Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

df_new = df.withColumnRenamed("old_name", "new_name") \
           .withColumn("price", col("price").cast("double"))

df_new.printSchema()
df_new.show()

root
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- base_price: long (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+----------+-----------+------+------------+----------+------+--------+
|product_id|   category| price|product_name|base_price|region|priority|
+----------+-----------+------+------------+----------+------+--------+
|       101|Electronics|1200.0|       Phone|      1000| North|    High|
|       102|  Furniture|2500.0|       Table|      2000| South|     Low|
|       103|Electronics| 800.0|      Laptop|       700|  West|  Medium|
|       104|   Clothing| 500.0|       Shirt|       400| North|    High|
|       105|Electronics|  NULL|     Speaker|      1500|  East|    High|
+----------+-----------+------+------------+----------+------+--------+



**
## Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 
**
ANS: \
Spark stores all transformations as a Lineage Graph (DAG).
If an executor fails:
- Lost partitions are recomputed using the DAG.
- No need to reload the entire dataset.
- This provides fault tolerance.

In [0]:
# Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

result = df_orders.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

result.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1500|
|       4|Completed|  2200|
+--------+---------+------+



**
## Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.
**
ANS: \
Predicate Pushdown means Spark sends filter conditions directly to Parquet files.
- Example:
df.filter(col("price") > 1000)
- Only matching data is read from disk.
- Benefits:\
Faster execution \
Less memory usage \
Less disk I/O

In [0]:
# Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

df_final = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

df_final.show()

+----------+-----------+-----+------------+----------+------+--------+-----------+
|product_id|   category|price|product_name|base_price|region|priority|final_price|
+----------+-----------+-----+------------+----------+------+--------+-----------+
|       101|Electronics| 1200|       Phone|      1000| North|    High|     1180.0|
|       102|  Furniture| 2500|       Table|      2000| South|     Low|     2360.0|
|       103|Electronics|  800|      Laptop|       700|  West|  Medium|      826.0|
|       104|   Clothing|  500|       Shirt|       400| North|    High|      472.0|
|       105|Electronics| NULL|     Speaker|      1500|  East|    High|     1770.0|
+----------+-----------+-----+------------+----------+------+--------+-----------+



**
## Q11: What is the difference between Transformations and Actions? Provide two examples of each. 
**

| Transformations  | Actions           |
| ---------------- | ----------------- |
| Lazy             | Trigger execution |
| Return DataFrame | Return result     |

Transformation Examples

- filter() 
- select()

Action Examples

- show() 
- count()

**
## Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 
**

df_parquet = spark.read.parquet("path/to/input")

result = df_parquet.filter(col("user_id").isNotNull())

result.write.mode("overwrite").option("header", True).csv("path/to/output")

**
## Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 
**

| Client Mode                  | Cluster Mode               |
| ---------------------------- | -------------------------- |
| Driver runs on local machine | Driver runs inside cluster |
| Good for development         | Good for production        |
| Client must stay connected   | Client can disconnect      |
| Less fault tolerant          | More fault tolerant        |



In [0]:
# Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

result = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

result.show()

+----------+-----------+-----+------------+----------+------+--------+
|product_id|   category|price|product_name|base_price|region|priority|
+----------+-----------+-----+------------+----------+------+--------+
|       101|Electronics| 1200|       Phone|      1000| North|    High|
|       104|   Clothing|  500|       Shirt|       400| North|    High|
|       105|Electronics| NULL|     Speaker|      1500|  East|    High|
+----------+-----------+-----+------------+----------+------+--------+



**
## Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?
**

- .show(5) displays only the first five rows, so it uses very little driver memory.
- .collect() transfers all rows from the executors to the driver. On very large datasets, this can consume excessive memory or even cause an OutOfMemoryError.

Best Practice: For exploring large datasets, use .show(), .limit(), or .take() instead of .collect().